In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
file_path = os.path.join("../data", "raw", "accepted_2007_to_2018Q4.csv")

In [3]:
print("Loading dataset...")

Loading dataset...


In [4]:
df = pd.read_csv(file_path, low_memory=False)

In [5]:
print("Dataset Loaded Successfully.")
print("\nShape of dataset:")
print(df.shape)

Dataset Loaded Successfully.

Shape of dataset:
(2260701, 151)


In [6]:
print("\nFirst 5 rows:")
print(df.head())


First 5 rows:
         id  member_id  loan_amnt  funded_amnt  funded_amnt_inv        term  \
0  68407277        NaN     3600.0       3600.0           3600.0   36 months   
1  68355089        NaN    24700.0      24700.0          24700.0   36 months   
2  68341763        NaN    20000.0      20000.0          20000.0   60 months   
3  66310712        NaN    35000.0      35000.0          35000.0   60 months   
4  68476807        NaN    10400.0      10400.0          10400.0   60 months   

   int_rate  installment grade sub_grade  ... hardship_payoff_balance_amount  \
0     13.99       123.03     C        C4  ...                            NaN   
1     11.99       820.28     C        C1  ...                            NaN   
2     10.78       432.66     B        B4  ...                            NaN   
3     14.85       829.90     C        C5  ...                            NaN   
4     22.45       289.91     F        F1  ...                            NaN   

  hardship_last_payment_amoun

In [7]:
print("\nUnique loan status values:")
print(df["loan_status"].value_counts())


Unique loan status values:
loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64


In [8]:
# Keep only final loan outcomes
df = df[df["loan_status"].isin(["Fully Paid", "Charged Off"])]

print("\nAfter filtering final loan outcomes:")
print(df["loan_status"].value_counts())

# Create binary target variable
df["default"] = df["loan_status"].apply(lambda x: 1 if x == "Charged Off" else 0)

print("\nTarget distribution:")
print(df["default"].value_counts())

print("\nNew shape after filtering:")
print(df.shape)


After filtering final loan outcomes:
loan_status
Fully Paid     1076751
Charged Off     268559
Name: count, dtype: int64

Target distribution:
default
0    1076751
1     268559
Name: count, dtype: int64

New shape after filtering:
(1345310, 152)


C:\Users\krish\AppData\Local\Temp\ipykernel_22136\4141087860.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["default"] = df["loan_status"].apply(lambda x: 1 if x == "Charged Off" else 0)


In [9]:
default_counts = df["default"].value_counts(normalize=True) * 100
print("\nClass distribution in percentage:")
print(default_counts)


Class distribution in percentage:
default
0    80.037389
1    19.962611
Name: proportion, dtype: float64


In [10]:
suspicious_keywords = [
    "payment",
    "recover",
    "collection",
    "last",
    "total",
    "settlement",
    "out_prncp",
    "funded_amnt_inv"
]

leakage_cols = [col for col in df.columns
                if any(keyword in col.lower() for keyword in suspicious_keywords)]

print("\nPotential leakage columns:")
for col in leakage_cols:
    print(col)


Potential leakage columns:
funded_amnt_inv
inq_last_6mths
mths_since_last_delinq
mths_since_last_record
total_acc
out_prncp
out_prncp_inv
total_pymnt
total_pymnt_inv
total_rec_prncp
total_rec_int
total_rec_late_fee
recoveries
collection_recovery_fee
last_pymnt_d
last_pymnt_amnt
last_credit_pull_d
last_fico_range_high
last_fico_range_low
collections_12_mths_ex_med
mths_since_last_major_derog
total_bal_il
total_rev_hi_lim
total_cu_tl
inq_last_12m
total_bal_ex_mort
total_bc_limit
total_il_high_credit_limit
sec_app_inq_last_6mths
sec_app_collections_12_mths_ex_med
sec_app_mths_since_last_major_derog
payment_plan_start_date
hardship_last_payment_amount
debt_settlement_flag
debt_settlement_flag_date
settlement_status
settlement_date
settlement_amount
settlement_percentage
settlement_term


In [11]:
true_leakage_cols = [
    "out_prncp",
    "out_prncp_inv",
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_d",
    "last_pymnt_amnt",
    "payment_plan_start_date",
    "debt_settlement_flag",
    "debt_settlement_flag_date",
    "settlement_status",
    "settlement_date",
    "settlement_amount",
    "settlement_percentage",
    "settlement_term"
]

print("True leakage columns count:", len(true_leakage_cols))

True leakage columns count: 19


In [12]:
df = df.drop(columns=true_leakage_cols, errors="ignore")

print("Shape after removing leakage:", df.shape)

Shape after removing leakage: (1345310, 133)


In [13]:
useless_cols = [
    "id",
    "member_id",
    "url",
    "desc",
    "title",
    "zip_code",
    "emp_title"
]

df = df.drop(columns=useless_cols, errors="ignore")

print("Shape after removing useless columns:", df.shape)

Shape after removing useless columns: (1345310, 126)


In [14]:
# Calculate missing values percentage
missing_percentage = df.isnull().mean() * 100

# Sort descending
missing_percentage = missing_percentage.sort_values(ascending=False)

print("\nTop 20 Columns With Highest Missing %:")
print(missing_percentage.head(20))


Top 20 Columns With Highest Missing %:
next_pymnt_d                                  100.000000
orig_projected_additional_accrued_interest     99.720585
hardship_loan_status                           99.572292
deferral_term                                  99.572292
hardship_amount                                99.572292
hardship_reason                                99.572292
hardship_payoff_balance_amount                 99.572292
hardship_status                                99.572292
hardship_type                                  99.572292
hardship_start_date                            99.572292
hardship_length                                99.572292
hardship_dpd                                   99.572292
hardship_last_payment_amount                   99.572292
hardship_end_date                              99.572292
sec_app_mths_since_last_major_derog            99.506062
sec_app_revol_util                             98.639570
revol_bal_joint                                9

In [15]:
# Identify columns with more than 80% missing
high_missing_cols = missing_percentage[missing_percentage > 80].index.tolist()

print("Columns to drop (missing > 80%):", len(high_missing_cols))

# Drop them
df = df.drop(columns=high_missing_cols)

print("Shape after dropping high-missing columns:", df.shape)

Columns to drop (missing > 80%): 31
Shape after dropping high-missing columns: (1345310, 95)


In [16]:
# Recalculate missing percentage after dropping high-missing columns
missing_percentage = df.isnull().mean() * 100
missing_percentage = missing_percentage.sort_values(ascending=False)

print("\nColumns with remaining missing values (> 0%):")
print(missing_percentage[missing_percentage > 0].head(20))


Columns with remaining missing values (> 0%):
mths_since_recent_bc_dlq          76.286506
mths_since_last_major_derog       73.704945
mths_since_recent_revol_delinq    66.553285
il_util                           65.434287
mths_since_rcnt_il                61.095807
all_util                          60.043038
inq_last_12m                      60.039173
total_cu_tl                       60.039173
open_acc_6m                       60.039173
total_bal_il                      60.039099
open_rv_24m                       60.039099
max_bal_bc                        60.039099
open_rv_12m                       60.039099
inq_fi                            60.039099
open_il_12m                       60.039099
open_act_il                       60.039099
open_il_24m                       60.039099
mths_since_last_delinq            50.452535
mths_since_recent_inq             12.939100
num_tl_120dpd_2m                   8.726688
dtype: float64


In [17]:
months_cols = [
    'mths_since_recent_bc_dlq',
    'mths_since_last_major_derog',
    'mths_since_recent_revol_delinq',
    'mths_since_last_delinq'
]

for col in months_cols:
    if col in df.columns:
        df[col + "_missing_flag"] = df[col].isnull().astype(int)
        df[col] = df[col].fillna(999)

print("Handled 'months since' columns")

Handled 'months since' columns


In [18]:
util_cols = [
    'il_util','all_util','inq_last_12m','total_cu_tl',
    'open_acc_6m','total_bal_il','open_rv_24m',
    'open_rv_12m','inq_fi','open_il_12m',
    'open_act_il','open_il_24m','max_bal_bc'
]

for col in util_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

print("Handled utilization columns")

Handled utilization columns


In [19]:
df['mths_since_recent_inq'] = df['mths_since_recent_inq'].fillna(
    df['mths_since_recent_inq'].median()
)

df['num_tl_120dpd_2m'] = df['num_tl_120dpd_2m'].fillna(
    df['num_tl_120dpd_2m'].median()
)

print("Handled small missing columns")

Handled small missing columns


In [20]:
print("Remaining missing %:")
print((df.isnull().mean() * 100).sort_values(ascending=False).head(10))

Remaining missing %:
mths_since_rcnt_il       61.095807
mo_sin_old_il_acct        7.847634
emp_length                5.835904
pct_tl_nvr_dlq            5.030885
avg_cur_bal               5.021073
mo_sin_old_rev_tl_op      5.019512
mo_sin_rcnt_rev_tl_op     5.019512
num_rev_accts             5.019512
mo_sin_rcnt_tl            5.019438
num_il_tl                 5.019438
dtype: float64


In [21]:
if 'mths_since_rcnt_il' in df.columns:
    df['mths_since_rcnt_il_missing_flag'] = df['mths_since_rcnt_il'].isnull().astype(int)
    df['mths_since_rcnt_il'] = df['mths_since_rcnt_il'].fillna(999)

print("Handled mths_since_rcnt_il")

Handled mths_since_rcnt_il


In [22]:
# Fill numeric columns with median
numeric_fill_cols = [
    'mo_sin_old_il_acct','pct_tl_nvr_dlq','avg_cur_bal',
    'mo_sin_old_rev_tl_op','mo_sin_rcnt_rev_tl_op',
    'num_rev_accts','mo_sin_rcnt_tl','num_il_tl'
]

for col in numeric_fill_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# Fill emp_length with mode
if 'emp_length' in df.columns:
    df['emp_length'] = df['emp_length'].fillna(df['emp_length'].mode()[0])

print("Handled remaining moderate missing columns")

Handled remaining moderate missing columns


In [23]:
print("Final missing percentage (top 10):")
print((df.isnull().mean() * 100).sort_values(ascending=False).head(10))

Final missing percentage (top 10):
num_tl_30dpd                  5.019438
total_rev_hi_lim              5.019438
tot_coll_amt                  5.019438
tot_cur_bal                   5.019438
num_tl_op_past_12m            5.019438
total_il_high_credit_limit    5.019438
num_bc_tl                     5.019438
num_accts_ever_120_pd         5.019438
num_actv_bc_tl                5.019438
num_tl_90g_dpd_24m            5.019438
dtype: float64


In [24]:
final_numeric_cols = [
    'num_tl_30dpd','total_rev_hi_lim','tot_coll_amt',
    'tot_cur_bal','num_tl_op_past_12m',
    'total_il_high_credit_limit','num_bc_tl',
    'num_accts_ever_120_pd','num_actv_bc_tl',
    'num_tl_90g_dpd_24m'
]

for col in final_numeric_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

print("Final numeric missing handled")

Final numeric missing handled


In [25]:
print("Remaining missing % after final cleaning:")
print((df.isnull().mean() * 100).sort_values(ascending=False).head(20))

Remaining missing % after final cleaning:
num_actv_rev_tl               5.019438
num_op_rev_tl                 5.019438
num_rev_tl_bal_gt_0           5.019438
tot_hi_cred_lim               5.019438
bc_util                       4.602062
percent_bc_gt_75              4.575525
bc_open_to_buy                4.544900
mths_since_recent_bc          4.476366
num_sats                      4.150791
num_bc_sats                   4.150791
acc_open_past_24mths          3.514506
total_bal_ex_mort             3.514506
total_bc_limit                3.514506
mort_acc                      3.514506
revol_util                    0.063703
pub_rec_bankruptcies          0.051810
dti                           0.027800
chargeoff_within_12_mths      0.004163
collections_12_mths_ex_med    0.004163
last_credit_pull_d            0.004088
dtype: float64


In [26]:

# Select numeric columns safely
numeric_cols = df.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# Select non-numeric columns safely
categorical_cols = df.select_dtypes(exclude=[np.number]).columns

for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Final cleanup done safely")

Final cleanup done safely


In [27]:
print("Total remaining missing values:")
print(df.isnull().sum().sum())

Total remaining missing values:
0


In [28]:
categorical_cols = df.select_dtypes(exclude=[np.number]).columns

print("Number of categorical columns:", len(categorical_cols))
print("\nCategorical Columns:\n")
print(categorical_cols)

Number of categorical columns: 17

Categorical Columns:

Index(['term', 'grade', 'sub_grade', 'emp_length', 'home_ownership',
       'verification_status', 'issue_d', 'loan_status', 'pymnt_plan',
       'purpose', 'addr_state', 'earliest_cr_line', 'initial_list_status',
       'last_credit_pull_d', 'application_type', 'hardship_flag',
       'disbursement_method'],
      dtype='str')


In [29]:
for col in categorical_cols:
    print(f"{col} --> {df[col].nunique()} unique values")

term --> 2 unique values
grade --> 7 unique values
sub_grade --> 35 unique values
emp_length --> 11 unique values
home_ownership --> 6 unique values
verification_status --> 3 unique values
issue_d --> 139 unique values
loan_status --> 2 unique values
pymnt_plan --> 1 unique values
purpose --> 14 unique values
addr_state --> 51 unique values
earliest_cr_line --> 739 unique values
initial_list_status --> 2 unique values
last_credit_pull_d --> 140 unique values
application_type --> 2 unique values
hardship_flag --> 1 unique values
disbursement_method --> 2 unique values


In [30]:
df = df.drop(columns=['loan_status','pymnt_plan','hardship_flag'])

print("Dropped useless categorical columns")
print("New shape:", df.shape)

Dropped useless categorical columns
New shape: (1345310, 97)


In [31]:


date_cols = ['issue_d','earliest_cr_line','last_credit_pull_d']

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], format='%b-%Y')

print("Converted date columns to datetime")

Converted date columns to datetime


In [32]:
reference_date = df['issue_d'].max()
print("Reference date:", reference_date)

Reference date: 2018-12-01 00:00:00


In [33]:
df['loan_age_months'] = (reference_date - df['issue_d']).dt.days // 30

In [34]:
df['credit_history_months'] = (reference_date - df['earliest_cr_line']).dt.days // 30

In [35]:
df['months_since_last_credit_pull'] = (reference_date - df['last_credit_pull_d']).dt.days // 30

In [36]:
df = df.drop(columns=['issue_d','earliest_cr_line','last_credit_pull_d'])

print("Dropped raw date columns")
print("New shape:", df.shape)

Dropped raw date columns
New shape: (1345310, 97)


In [37]:
categorical_cols = df.select_dtypes(exclude=[np.number]).columns

print("Categorical columns remaining:", len(categorical_cols))
print(categorical_cols)

Categorical columns remaining: 11
Index(['term', 'grade', 'sub_grade', 'emp_length', 'home_ownership',
       'verification_status', 'purpose', 'addr_state', 'initial_list_status',
       'application_type', 'disbursement_method'],
      dtype='str')


In [38]:
binary_cols = ['term','initial_list_status','application_type','disbursement_method']

for col in binary_cols:
    if col in df.columns:
        print(col, df[col].unique())

term <StringArray>
[' 36 months', ' 60 months']
Length: 2, dtype: str
initial_list_status <StringArray>
['w', 'f']
Length: 2, dtype: str
application_type <StringArray>
['Individual', 'Joint App']
Length: 2, dtype: str
disbursement_method <StringArray>
['Cash', 'DirectPay']
Length: 2, dtype: str


In [39]:
df['term'] = df['term'].str.strip()  # remove leading space

df['term'] = df['term'].map({
    '36 months': 0,
    '60 months': 1
})

df['initial_list_status'] = df['initial_list_status'].map({
    'f': 0,
    'w': 1
})

df['application_type'] = df['application_type'].map({
    'Individual': 0,
    'Joint App': 1
})

df['disbursement_method'] = df['disbursement_method'].map({
    'Cash': 0,
    'DirectPay': 1
})

print("Binary encoding completed")

Binary encoding completed


In [40]:
print(df[['term','initial_list_status','application_type','disbursement_method']].head())

   term  initial_list_status  application_type  disbursement_method
0     0                    1                 0                    0
1     0                    1                 0                    0
2     1                    1                 1                    0
4     1                    1                 0                    0
5     0                    1                 0                    0


In [41]:
categorical_remaining = [
    'grade','sub_grade','emp_length',
    'home_ownership','verification_status',
    'purpose','addr_state'
]

df = pd.get_dummies(
    df,
    columns=categorical_remaining,
    drop_first=True
)

print("One-hot encoding completed")
print("New shape:", df.shape)

One-hot encoding completed
New shape: (1345310, 210)


In [42]:
non_numeric = df.select_dtypes(exclude=[np.number]).columns

print("Non-numeric columns remaining:", len(non_numeric))
print(non_numeric)

Non-numeric columns remaining: 120
Index(['grade_B', 'grade_C', 'grade_D', 'grade_E', 'grade_F', 'grade_G',
       'sub_grade_A2', 'sub_grade_A3', 'sub_grade_A4', 'sub_grade_A5',
       ...
       'addr_state_SD', 'addr_state_TN', 'addr_state_TX', 'addr_state_UT',
       'addr_state_VA', 'addr_state_VT', 'addr_state_WA', 'addr_state_WI',
       'addr_state_WV', 'addr_state_WY'],
      dtype='str', length=120)


In [43]:
print("Target column exists:", 'default' in df.columns)

Target column exists: True


In [44]:
print(df['default'].value_counts())

default
0    1076751
1     268559
Name: count, dtype: int64


In [45]:
print("Final dataset shape:", df.shape)

Final dataset shape: (1345310, 210)


In [46]:
print(df.dtypes.value_counts())

bool       120
float64     77
int64       13
Name: count, dtype: int64


In [47]:
print("Object dtype columns:")
print(df.select_dtypes(include=['object']).columns)

Object dtype columns:
Index([], dtype='str')


In [48]:
import os
print(os.getcwd())

E:\Skill Sphere\Artificial Intelligence\Projects\loan-risk-system\notebooks


In [52]:
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/cleaned_credit_data.csv", index=False)